# 14 - Machine learning from YOLO inferences

Convert detection-level predictions into one row per frame, attach experimental labels, and train a standard scikit-learn model.


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from vision.yolo.ml import frame_features, merge_labels, prepare_xy


In [ ]:
# Replace this with predictions returned by predict_video or track_video.
predictions = pd.read_parquet("notebooks/outputs/predictions.parquet")
features = frame_features(predictions)
features.head()


In [ ]:
# Example label table. In a real experiment this can come from manual scoring,
# treatment metadata, behavioral epochs, genotype, condition, etc.
labels = pd.DataFrame({
    "frame": features["frame"],
    "condition": (features["frame"] % 2).astype(int),
})

dataset = merge_labels(features, labels, on="frame")
X, y = prepare_xy(dataset, target="condition", drop_columns=["frame", "timestamp"])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)
predicted = model.predict(X_test)
print(classification_report(y_test, predicted))


## SHAP for the downstream model
Install the optional explainability dependencies with `pip install -e ".[explain]"`.


In [ ]:
# from vision.yolo.ml import shap_values
# explanation = shap_values(model, X_train, max_samples=200)
# import shap
# shap.plots.beeswarm(explanation)
